# Lab 5 · Long-Term Memory

Short-term memory recalls the current conversation. **Long-term memory** is the durable layer: entities, facts, and preferences that outlive any single session and are meant to be recalled weeks later, in a brand-new conversation.

`neo4j-agent-memory` gives you three explicit write methods:
- `add_entity` for a thing worth remembering, such as a company or a person, with deduplication
- `add_fact` for a subject, predicate, object statement
- `add_preference` for how the user likes to work

This notebook writes some durable knowledge, then opens a fresh session and shows it coming back. That is the proof the knowledge is not tied to a conversation.

**Learning objectives**
- Persist entities, facts, and preferences with the long-term memory API
- Retrieve them in a new session with `search_entities` and `long_term.get_context`
- Optionally adopt the existing SEC 10-K `Company` nodes as long-term entities
- See the `Entity`, `Fact`, and `Preference` nodes in Neo4j

**Prerequisites:** same as the short-term notebook, the Lab 1 seed load and a populated `CONFIG.txt`.

In [ ]:
%pip install "neo4j-agent-memory[bedrock]==0.5.0" strands-agents "neo4j-graphrag[bedrock]>=1.18.0" -q

## Setup

The same async memory client as the short-term notebook, over the same Aura instance and Titan embeddings. Long-term memory does not need the agent, so this notebook stays focused on the memory API itself.

In [ ]:
import sys
from pathlib import Path

# Lab 5 lives beside Lab 4. Add both to the import path:
#   this lab's dir   provides lib.data_utils and lib.memory_utils
#   Lab 4's lib dir  provides graphrag_agent, the canonical GraphRAG agent
#                    module, imported here rather than copied so there is one
#                    source of truth.
LAB5_DIR = Path.cwd()
LAB4_LIB = LAB5_DIR.parent / "Lab_4_GraphRAG_Agent" / "lib"
for p in (str(LAB4_LIB), str(LAB5_DIR)):
    if p not in sys.path:
        sys.path.insert(0, p)

print(f"Lab 5 dir:  {LAB5_DIR}")
print(f"Lab 4 lib:  {LAB4_LIB}")

In [ ]:
from lib.memory_utils import build_memory_client

memory = build_memory_client()
await memory.connect()
print("Memory client connected.")

## Write durable knowledge

Three kinds of long-term memory. `add_entity` returns an `(entity, dedup_result)` tuple, where the dedup result tells you whether the entity was newly created or merged into an existing one. `add_fact` takes its object via the `obj=` argument. `add_preference` records how the user wants answers shaped.

In [ ]:
from neo4j_agent_memory.memory.long_term import EntityType

# An entity worth remembering across sessions.
entity, dedup = await memory.long_term.add_entity(
    name="Apple",
    entity_type=EntityType.ORGANIZATION,
    description="Consumer electronics and services company; SEC 10-K filer.",
)
print(f"Entity: {entity.name} ({entity.type}) - dedup action: {dedup.action}")

# A fact: subject, predicate, object.
fact = await memory.long_term.add_fact(
    subject="Apple",
    predicate="COMPETES_WITH",
    obj="Microsoft",
)
print(f"Fact: {fact.subject} {fact.predicate} {fact.object}")

# A preference: how this user likes their answers.
pref = await memory.long_term.add_preference(
    category="analysis",
    preference="Prefers concise risk summaries as bullet points.",
)
print(f"Preference [{pref.category}]: {pref.preference}")

## Recall in a fresh session

The knowledge above was written with no `session_id`, so it belongs to the store rather than to a conversation. A brand-new session can still find it. `search_entities` does a semantic lookup, and `long_term.get_context` formats the durable knowledge for an LLM prompt.

Because `search_entities` ranks by vector similarity, it takes a `threshold`. A topical query rarely scores as high as an exact name, so we lower the threshold below its 0.7 default to accept a semantically related match.

In [ ]:
# search_entities is a semantic (vector) lookup, so the topical query
# "technology company" matches the Apple entity by meaning, not by exact name.
# Titan similarity for a topical match lands near 0.65, so we pass a threshold
# below the 0.7 default to accept it. Lower the threshold to recall more
# loosely related entities; raise it to demand a closer match.
entities = await memory.long_term.search_entities(
    "technology company", limit=5, threshold=0.5
)
print("search_entities('technology company'):")
for e in entities:
    print(f"  - {e.name} ({e.type})")

print("\nlong_term.get_context('What do we know about Apple?'):")
print(await memory.long_term.get_context("What do we know about Apple?"))

## Optional: adopt the existing SEC graph

This cell is optional. Skip it to keep the core path short.

The graph already holds `Company` nodes from the SEC 10-K seed load. `adopt_existing_graph` layers long-term memory over those existing nodes instead of creating parallel `Entity` nodes, so when a message later mentions "Apple" it links to the company you already have rather than a duplicate. The operation is idempotent.

In [ ]:
report = await memory.schema.adopt_existing_graph(
    label_to_type={"Company": "ORGANIZATION"},
    name_property_per_label={"Company": "name"},
)
print(
    f"Adopted {report.total_migrated} nodes "
    f"({report.total_already_adopted} already adopted, {report.total_skipped} skipped)."
)

## Inspect the memory in Neo4j

The durable knowledge is now queryable alongside the SEC 10-K graph.

In [ ]:
for label in ("Entity", "Fact", "Preference"):
    rows = await memory.query.cypher(f"MATCH (n:{label}) RETURN count(n) AS count")
    print(f"{label}: {rows[0]['count']} nodes")

print("\nSample entities:")
rows = await memory.query.cypher(
    "MATCH (e:Entity) RETURN e.name AS name, e.type AS type LIMIT 10"
)
for r in rows:
    print(f"  - {r['name']} ({r['type']})")

## Cleanup

Close the memory client.

In [ ]:
await memory.close()
print("Closed memory client.")

## Summary

You persisted durable knowledge and recalled it from a fresh session:

| Method | Writes | Recalled by |
|--------|--------|-------------|
| `add_entity` | An `Entity` node, deduplicated | `search_entities` |
| `add_fact` | A subject, predicate, object `Fact` | `long_term.get_context` |
| `add_preference` | A `Preference` node | `long_term.get_context` |

Together with the short-term layer from the previous notebook, the agent now has both conversational recall and durable, cross-session knowledge, all stored in the same Neo4j graph as the SEC 10-K data.

**Going further:** the workshop site page describes a third pillar, reasoning traces, as the observability and audit layer over an agent's memory.

---

**Next:** [Lab 6: Neo4j MCP Server](../Lab_6_MCP_Server/), serving graph retrieval as remote tools over MCP